# Phase 1: Data Wrangling

In [ ]:
import pandas as pd
import gc

# 1.1 Read Data Using Chunking
chunk_list = []
for chunk in pd.read_csv('33k/postings.csv', chunksize=100_000):
    chunk_list.append(chunk)
    gc.collect()

df_jobs = pd.concat(chunk_list, ignore_index=True)
del chunk_list
gc.collect()

df_skills = pd.read_csv('33k/jobs/job_skills.csv')
df_industries = pd.read_csv('33k/jobs/job_industries.csv')

print(f"job_postings shape: {df_jobs.shape}")
print(f"job_skills shape: {df_skills.shape}")
print(f"job_industries shape: {df_industries.shape}")

In [ ]:
# 1.2 Data Assessment
missing = df_jobs.isnull().sum().reset_index()
missing.columns = ['column', 'missing_count']
missing['missing_pct'] = (missing['missing_count'] / len(df_jobs) * 100).round(2)
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print("=== MISSING VALUES ===")
print(missing.to_string())

print(f"\n=== DUPLICATES ===")
print(f"Total duplicate rows: {df_jobs.duplicated().sum()}")
print(f"Duplicate job_id: {df_jobs.duplicated(subset=['job_id']).sum()}")

print("\n=== DATA TYPES ===")
print(df_jobs.dtypes)

print("\n=== BASIC STATISTICS ===")
print(df_jobs.describe())

In [ ]:
# 1.3 Data Cleaning
# 1. Remove duplicate job_id
df_jobs.drop_duplicates(subset=['job_id'], keep='first', inplace=True)

# 2. Drop rows missing critical columns
df_jobs.dropna(subset=['description', 'title'], inplace=True)

# 3. Strip whitespace from text columns
df_jobs['title'] = df_jobs['title'].str.strip()
df_jobs['description'] = df_jobs['description'].str.strip()

# 4. Standardize data types
df_jobs['job_id'] = df_jobs['job_id'].astype(str)
df_jobs['company_id'] = df_jobs['company_id'].astype(str)

# 5. Handle salary nulls & remote nulls
df_jobs['max_salary'] = df_jobs['max_salary'].fillna(df_jobs['max_salary'].median())
df_jobs['min_salary'] = df_jobs['min_salary'].fillna(df_jobs['min_salary'].median())
# Assume missing remote_allowed means not remote (0)
df_jobs['remote_allowed'] = df_jobs['remote_allowed'].fillna(0.0)

# 6. Reset index
df_jobs.reset_index(drop=True, inplace=True)

print(f"Shape after cleaning: {df_jobs.shape}")

# Phase 2: Text Preprocessing (NLP Pipeline)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def full_nlp_pipeline(text: str) -> str:
    """
    Full NLP pipeline:
    1. Noise Removal (HTML, URLs, numbers, special chars)
    2. Case Folding (lowercase)
    3. Tokenization
    4. Stopword Removal
    5. Lemmatization
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''

    # Step 1: Noise Removal
    text = re.sub(r'<[^>]+>', ' ', text)            # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # Remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)             # Remove emails
    text = re.sub(r'\d+', ' ', text)                 # Remove numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)         # Remove symbols
    text = re.sub(r'\s+', ' ', text).strip()         # Normalize whitespace

    # Step 2: Case Folding
    text = text.lower()

    # Step 3: Tokenization
    tokens = text.split()

    # Step 4: Stopword Removal (keep only meaningful words, min length 3)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    # Step 5: Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

# Apply pipeline
print("Running NLP pipeline on descriptions...")
df_jobs['clean_description'] = df_jobs['description'].apply(full_nlp_pipeline)

# Remove rows where clean_description ended up empty
df_jobs = df_jobs[df_jobs['clean_description'].str.len() > 10]
df_jobs.reset_index(drop=True, inplace=True)

print(f"Shape after NLP: {df_jobs.shape}")

In [ ]:
# Show 3 before/after examples
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"BEFORE: {df_jobs['description'].iloc[i][:300]}")
    print(f"AFTER : {df_jobs['clean_description'].iloc[i][:300]}")

# Phase 3: Feature Engineering (Vectorization)

In [ ]:
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import pickle

os.makedirs('output', exist_ok=True)

# 3.1 TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df_jobs['clean_description'])
feature_names = tfidf.get_feature_names_out()

save_npz('output/tfidf_matrix.npz', tfidf_matrix)
with open('output/tfidf_feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Top 30 features have been calculated.")

In [ ]:
# 3.2 Semantic Embedding
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
descriptions = df_jobs['clean_description'].tolist()
BATCH_SIZE = 512
all_embeddings = []

for i in range(0, len(descriptions), BATCH_SIZE):
    batch = descriptions[i:i + BATCH_SIZE]
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.append(batch_embeddings)

final_embeddings = np.vstack(all_embeddings)
np.save('output/job_embeddings.npy', final_embeddings)

print(f"Embedding matrix shape: {final_embeddings.shape}")

In [ ]:
# 3.3 Save Final Dataset
cols_to_keep = ['job_id', 'company_id', 'title', 'description', 'clean_description',
                'max_salary', 'min_salary', 'location', 'formatted_experience_level',
                'applies', 'views', 'remote_allowed', 'job_posting_url']

available_cols = [c for c in cols_to_keep if c in df_jobs.columns]
df_final = df_jobs[available_cols].copy()
df_final.to_csv('output/linkedin_jobs_cleaned.csv', index=False)

print(f"Final CSV saved with shape: {df_final.shape}")
print("Row alignment check:", len(df_final) == tfidf_matrix.shape[0] == final_embeddings.shape[0])

# Phase 4: Save Data Dictionary

In [ ]:
data_dict_md = """# Data Dictionary — LinkedIn Jobs Cleaned Dataset
**Project:** Hirings (CC26-PRU419)
**Prepared by:** Arvin Demas Naryama
**Last Updated:** April 2026
**Source:** LinkedIn Job Postings 2024 (Kaggle, ~123K rows from 33k/postings.csv)

---

## File: linkedin_jobs_cleaned.csv
Total rows: 123.824 | Total columns: 13

| Column | Data Type | Description | Example Value | Category |
|--------|-----------|-------------|---------------|----------|
| job_id | string | Unique LinkedIn job posting ID | "3901987" | Original |
| company_id | string | LinkedIn company ID | "1009" | Original |
| title | string | Job position title | "Senior Data Engineer" | Original |
| description | string | Raw job description text | "We are looking for a..." | Original |
| clean_description | string | Description after full NLP pipeline (noise removal, lowercase, stopword removal, lemmatization) | "look data engineer python skill" | Engineered |
| max_salary | float64 | Maximum annual salary in USD. Null values filled with column median. | 150000.0 | Original |
| min_salary | float64 | Minimum annual salary in USD. Null values filled with column median. | 80000.0 | Original |
| location | string | Job location in City, State format | "New York, NY" | Original |
| formatted_experience_level | string | Seniority level of the role | "Mid-Senior level" | Original |
| applies | float64 | Number of applications submitted. Null filled with 0. | 342.0 | Original |
| views | float64 | Number of times the posting was viewed | 1200.0 | Original |
| remote_allowed | float64 | Remote work flag: 1=remote allowed, 0=not remote. Null filled with 0. | 1.0 | Original |
| job_posting_url | string | Direct URL to the LinkedIn job posting | "https://linkedin.com/jobs/view/..." | Original |

---

## File: tfidf_matrix.npz
- **Type:** Scipy sparse matrix (CSR format)
- **Shape:** (123824, 5000)
- **Description:** TF-IDF weighted matrix from clean_description. Each row = one job posting. Each column = one of 5000 n-gram features (unigrams + bigrams).
- **Parameters used:** max_features=5000, ngram_range=(1,2), min_df=5, max_df=0.85, sublinear_tf=True
- **Top features:** work, team, service, customer, skill, job, company, opportunity, ability, year
- **How to load:**
```python
from scipy.sparse import load_npz
tfidf_matrix = load_npz('output/tfidf_matrix.npz')
```

---

## File: tfidf_feature_names.pkl
- **Type:** Python list (pickle format)
- **Length:** 5000
- **Description:** List of all 5000 feature names (words/bigrams) corresponding to columns in tfidf_matrix.npz
- **How to load:**
```python
import pickle
with open('output/tfidf_feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
```

---

## File: job_embeddings.npy
- **Type:** NumPy ndarray (float32)
- **Shape:** (123824, 384)
- **Description:** Semantic embedding vectors generated from clean_description using SentenceTransformer model all-MiniLM-L6-v2. Each row = one job posting. 384 dimensions capture semantic meaning.
- **Note:** Generate ulang di Google Colab menggunakan GPU untuk hasil embedding yang valid (kode murni tersedia di notebook_wrangling.ipynb).
- **How to load:**
```python
import numpy as np
embeddings = np.load('output/job_embeddings.npy')
```

---

## NLP Pipeline Summary
Steps applied to `description` column to produce `clean_description`:

| Step | Method | Library | Description |
|------|--------|---------|-------------|
| 1 | Noise Removal | re (Regex) | Removed HTML tags, URLs, emails, digits, special characters |
| 2 | Case Folding | Python built-in | Converted all text to lowercase |
| 3 | Tokenization | str.split() | Split sentences into individual word tokens |
| 4 | Stopword Removal | NLTK | Removed English stopwords + tokens shorter than 3 characters |
| 5 | Lemmatization | NLTK WordNetLemmatizer | Reduced words to their base/root form |

---

## Data Quality Summary

| Metric | Before Cleaning | After Cleaning |
|--------|----------------|----------------|
| Total rows | 123.849 | 123.824 |
| Duplicate rows | 0 | 0 |
| Null in description | 7 | 0 |
| Null in title | 0 | 0 |
| Null in max_salary | 75.94% | 0% (filled with median) |
| Null in min_salary | 75.94% | 0% (filled with median) |
| Null in remote_allowed | 87.69% | 0% (filled with 0) |
| Rows dropped (NLP empty) | - | 18 |
"""

with open('output/data_dictionary.md', 'w', encoding='utf-8') as f:
    f.write(data_dict_md)

print("Data Dictionary saved to output/data_dictionary.md!")

# Phase 5: Final Validation & Handoff Checklist

In [ ]:
import os
import numpy as np
from scipy.sparse import load_npz
import pandas as pd
import pickle

print("=" * 55)
print("   FINAL HANDOFF VALIDATION — CC26-PRU419 HIRINGS")
print("=" * 55)

# 1. CSV
df_check = pd.read_csv('output/linkedin_jobs_cleaned.csv')
print(f"\n✅ linkedin_jobs_cleaned.csv")
print(f"   Rows    : {len(df_check):,}")
print(f"   Columns : {df_check.shape}")
print(f"   Null in clean_description: {df_check['clean_description'].isnull().sum()}")

# 2. TF-IDF
tfidf_check = load_npz('output/tfidf_matrix.npz')
print(f"\n✅ tfidf_matrix.npz")
print(f"   Shape   : {tfidf_check.shape}")

# 3. Feature names
with open('output/tfidf_feature_names.pkl', 'rb') as f:
    fn = pickle.load(f)
print(f"\n✅ tfidf_feature_names.pkl")
print(f"   Total features: {len(fn)}")

# 4. Embeddings
emb_check = np.load('output/job_embeddings.npy')
print(f"\n✅ job_embeddings.npy")
print(f"   Shape   : {emb_check.shape}")

# 5. Data dictionary
dd_size = os.path.getsize('output/data_dictionary.md') / 1024
print(f"\n✅ data_dictionary.md")
print(f"   Size    : {dd_size:.1f} KB")

# 6. Row alignment
assert len(df_check) == tfidf_check.shape[0] == emb_check.shape[0], \
    "❌ MISMATCH! Fix before handoff."
print(f"\n✅ Row alignment: {len(df_check):,} rows — ALIGNED")

# 7. File sizes
print(f"\n📁 Output folder contents:")
for fname in os.listdir('output/'):
    fpath = f'output/{fname}'
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"   {fname}: {size_mb:.2f} MB")

print("\n" + "=" * 55)
print("✅ ALL CHECKS PASSED")
print("✅ DATASET READY FOR HANDOFF TO AINUR & ALYA")
print(f"✅ Deadline: April 27, 2026 — ON TRACK")
print("=" * 55)
